In [ ]:
import pypsa
import pandas as pd

import matplotlib.pyplot as plt
from _helpers_notebooks import load_config

### Snakemake

In [ ]:
if "snakemake" not in globals():
    from _helpers_notebooks import mock_snakemake
    snakemake = mock_snakemake(
            "plot_hourly",
    )

### Read product model

In [ ]:
n = pypsa.Network(snakemake.input.network)

In [ ]:
n.statistics()

### Hourly plot (clean)

In [ ]:
df = n.statistics.energy_balance(aggregate_time=False).loc[
    :,
    [
        "renewable_electricity", # battery exchange with renewables
        "direct_reduction_furnace",
        "electrolysis",
        "renewable_solar",
        "renewable_onwind",
        "battery_elec",
    ],
    "renewable_electricity",
]

df = df.loc[:, pd.Timestamp("2050-07-21") : pd.Timestamp("2050-07-26")]

df = (
    df.reset_index(level=["component", "bus_carrier"], drop=True)
    .groupby("carrier")
    .sum()
)

In [ ]:
order = ["renewable_onwind","renewable_solar","battery_elec",
         "renewable_electricity","direct_reduction_furnace",
        "electrolysis"]
df =df.loc[order, :]

In [ ]:
colors = snakemake.config["colors"]

In [ ]:
df_plot = df.T

df_plot.rename(
    columns={
        "battery_elec": "battery (discharging)",
        "electrolysis": "electrolysis",
        "direct_reduction_furnace": "direct reduction furnace",
        "renewable_solar": "solar",
        "renewable_onwind": "onshore wind",
        "renewable_electricity": "battery (charging)",
    },
    inplace=True,
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

df_plot.plot.area(ax=ax, stacked=True, color=colors, linewidth=0)

# Optional: overlay hydrogen consumption (as before)
n.statistics.energy_balance(aggregate_time=False).loc[
    "Link", "direct_reduction_furnace", "hydrogen"
].loc[pd.Timestamp("2050-07-21") : pd.Timestamp("2050-07-26")].plot(
    ax=ax, color="black", linewidth=1.5, label="DRI H$_2$ consumption"
)

# Styling (matching your original)
ax.grid(True, alpha=0.5, linestyle="--")
ax.set_xlabel("Day") #, fontsize=9)
ax.set_ylabel("Electricity Balance in MW")

ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)

plt.tight_layout()
plt.savefig(snakemake.output.hourly_plot, dpi=300, bbox_inches="tight")
plt.show()

### Plots in hourly resolution

In [ ]:
# n.statistics.energy_balance.iplot.area(bus_carrier="hydrogen")

#### All carriers balance

In [ ]:
# n.statistics.energy_balance.iplot.area()  # Ely als negativen Stromverbraucher, H2 Verbrauch der DRI als linie

### Part load behaviour

In [ ]:
# n.statistics.capacity_factor.plot.bar()

In [ ]:
n.links.p_min_pu

No part load or minimum load requirement set in this case

### Energy balance

In [ ]:
n.statistics.energy_balance().div(1e3)

In [ ]:
# # Electricity to steel ratio
# steel_demand = (
#     n.statistics.energy_balance().loc[:, "Steel", "Steel"].values[0]
# )  # in t of steel
# pv_supply = (
#     n.statistics.energy_balance().loc[:, "Photovoltaics", "Electricity"].values[0]
# )  # in MWh
# wind_supply = (
#     n.statistics.energy_balance().loc[:, "Wind energy", "Electricity"].values[0]
# )  # in MWh

# steel_demand
# elec_to_steel = (pv_supply + wind_supply) * (-1) / steel_demand  # in MWh/t
# elec_to_steel

### Misc

In [ ]:
n.statistics.energy_balance().loc["Load", "Steel", :].iloc[0]  # tonnes of steel

In [ ]:
n.statistics.optimal_capacity().mul(1e3) / n.statistics.energy_balance().loc[
    "Load", "Steel", :
].iloc[0]